# NCKH 01 — Dataset and Experimental Setup

This notebook produces the Section 4.1 figures for the selected
multiclass LightGBM experiment. The unit of analysis is `(SKU, t0)`.
Features use transactions available through `t0`; future sales are
used only to construct the evaluation label.

The target is a sales-based proxy for slow-moving risk. Online Retail II
does not contain stock-on-hand, stockout, cost or margin, so the figures
must not be interpreted as direct inventory evidence.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

ROOT = Path.cwd()
if not (ROOT / "reports" / "nckh_online_retail_multiclass").exists():
    ROOT = Path("..").resolve()
REPORT = ROOT / "reports" / "nckh_online_retail_multiclass"
SELECTED = REPORT / "selected"
FIG_DIR = REPORT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["CRITICAL", "HIGH_RISK", "MEDIUM_RISK", "LOW_RISK"]
CLASS_COLORS = {
    "CRITICAL": "#B42318",
    "HIGH_RISK": "#E76F00",
    "MEDIUM_RISK": "#176B87",
    "LOW_RISK": "#2A9D8F",
}
MODEL_LABELS = {
    "recent_zero_rule": "Recent-zero rule",
    "recent_quantile_rule": "Recent-sales quantile rule",
    "logistic_regression": "Logistic regression",
    "random_forest": "Random forest",
    "lightgbm": "LightGBM",
}
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

with open(SELECTED / "dataset_profile.json", encoding="utf-8") as handle:
    profile = json.load(handle)
with open(REPORT / "selected_config.json", encoding="utf-8") as handle:
    selected_config = json.load(handle)
with open(SELECTED / "metrics.json", encoding="utf-8") as handle:
    metrics = json.load(handle)

In [ ]:
snapshots = pd.read_parquet(SELECTED / "snapshots.parquet")
split = json.loads((SELECTED / "temporal_split.json").read_text(encoding="utf-8"))
label_definition = json.loads((SELECTED / "label_definition.json").read_text(encoding="utf-8"))
audit = profile["cleaning_audit"]
selected_horizon = selected_config["selected_horizon_days"]
print(f"Selected horizon: {selected_horizon} days")
print(f"Snapshot rows: {len(snapshots):,}")
print(f"Snapshot SKUs: {snapshots['sku'].nunique():,}")
print(f"Cutoffs: {snapshots['cutoff_date'].nunique()}")

## Table 1 — Cleaning audit

In [ ]:
cleaning_rows = [
    ("Raw transaction rows", audit["raw_rows"]),
    ("Removed: missing identifier/date", audit["removed_missing_identifier_or_datetime"]),
    ("Removed: cancelled invoices", audit["removed_cancelled_invoice_rows"]),
    ("Removed: non-positive quantity", audit["removed_non_positive_quantity_rows"]),
    ("Removed: non-positive price", audit["removed_non_positive_price_rows"]),
    ("Removed: non-product StockCode", audit["removed_non_product_stockcode_rows"]),
    ("Removed: exact duplicates", audit["removed_exact_duplicate_rows_after_filtering"]),
    ("Retained valid transactions", audit["retained_transaction_rows"]),
    ("Retained SKUs", audit["retained_skus"]),
]
cleaning_table = pd.DataFrame(cleaning_rows, columns=["Metric", "Value"])
cleaning_table

In [ ]:
removed = [
    audit["removed_missing_identifier_or_datetime"],
    audit["removed_cancelled_invoice_rows"],
    audit["removed_non_positive_quantity_rows"],
    audit["removed_non_positive_price_rows"],
    audit["removed_non_product_stockcode_rows"],
    audit["removed_exact_duplicate_rows_after_filtering"],
]
labels = [
    "Missing key/date", "Cancelled", "Quantity <= 0", "Price <= 0",
    "Non-product code", "Exact duplicate",
]
fig, ax = plt.subplots(figsize=(8.6, 4.8))
bars = ax.bar(labels, removed, color="#264653")
ax.set_title("Cleaning removals by rule")
ax.set_ylabel("Rows removed")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.2)
for bar, value in zip(bars, removed):
    ax.text(bar.get_x() + bar.get_width()/2, value, f"{value:,}",
            ha="center", va="bottom", fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "01_cleaning_waterfall.png", bbox_inches="tight")
plt.show()

## Figure 2 — Observation timeline and decision cutoffs

In [ ]:
cutoff_counts = snapshots.groupby("cutoff_date").size().reset_index(name="snapshot_rows")
cutoff_counts["cutoff_date"] = pd.to_datetime(cutoff_counts["cutoff_date"])
train_end = pd.Timestamp(split["train_cutoff_dates"][-1])
val_end = pd.Timestamp(split["validation_cutoff_dates"][-1])
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(cutoff_counts["cutoff_date"], cutoff_counts["snapshot_rows"],
        marker="o", color="#176B87", linewidth=2)
ax.axvspan(cutoff_counts["cutoff_date"].min(), train_end,
           color="#2A9D8F", alpha=0.12, label="Train")
ax.axvspan(train_end, val_end, color="#E9C46A", alpha=0.20, label="Validation")
ax.axvspan(val_end, cutoff_counts["cutoff_date"].max(),
           color="#E76F00", alpha=0.14, label="Test")
ax.set_title("SKU–cutoff snapshots and chronological split")
ax.set_ylabel("Number of eligible SKU snapshots")
ax.set_xlabel("Decision cutoff date")
ax.legend(frameon=False, ncol=3)
ax.grid(axis="y", alpha=0.2)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIG_DIR / "02_temporal_split_timeline.png", bbox_inches="tight")
plt.show()

## Figure 3 — Four-class label distribution

In [ ]:
prevalence = (
    snapshots["risk_class"].value_counts()
    .reindex(CLASS_NAMES, fill_value=0)
    .rename("count").to_frame()
)
prevalence["share"] = prevalence["count"] / prevalence["count"].sum()
prevalence

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.8))
bars = ax.bar(
    prevalence.index, prevalence["share"],
    color=[CLASS_COLORS[name] for name in CLASS_NAMES],
)
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_title(f"Risk-label distribution ({selected_horizon}-day horizon)")
ax.set_ylabel("Share of SKU–cutoff snapshots")
ax.set_xlabel("Risk class")
ax.grid(axis="y", alpha=0.2)
for bar, share, count in zip(bars, prevalence["share"], prevalence["count"]):
    ax.text(bar.get_x() + bar.get_width()/2, share,
            f"{share:.1%}\n(n={count:,})", ha="center", va="bottom", fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "03_risk_class_distribution.png", bbox_inches="tight")
plt.show()

## Table 2 — Experimental configuration

In [ ]:
config_table = pd.DataFrame([
    ("Source", profile["source_file"]),
    ("Lookback window", f"{selected_config['label_policy']['config']['lookback_days']} days"),
    ("Selected future horizon", f"{selected_horizon} days"),
    ("Cutoff frequency", f"{selected_config['label_policy']['config']['cutoff_frequency_days']} days"),
    ("Eligibility", "Historical quantity >= 5"),
    ("Split", "15/4/4 chronological cutoffs; actual row shares are reported separately"),
    ("Primary model", "Class-balanced multiclass LightGBM"),
    ("Label thresholds", "CRITICAL = zero; positive q25/q50 learned on train"),
    ("Random seed", str(selected_config["label_policy"]["config"]["random_state"])),
], columns=["Setting", "Value"])
config_table